# ANN replication: Stage A
- network development (network.py) (stage A)


In [1]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

DATA_NPZ = ROOT / "data/processed/prepared.npz"
MODEL_DIR = ROOT / "models/ann_baseline"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

from sandwich.ann.network import (INPUT_COLS, LAYERS, N_PARAMS, OUTPUT_COLS, SHAPES, 
                                  forward, init_weights, load_split, mse)

SEED = 42

In [2]:
data = load_split(DATA_NPZ)

assert data["X_train"].shape == (7200, 9) and data["Y_train"].shape == (7200, 8)
assert data["X_val"].shape == (900, 9) and data["X_test"].shape == (900, 9)

print("inputs:", INPUT_COLS)
print("outputs:", OUTPUT_COLS)
for part in ("train", "val", "test"):
    X, Y = data[f"X_{part}"], data[f"Y_{part}"]
    print(f"{part:<5}  {len(X):>5} rows   X in [{X.min():.3f}, {X.max():.3f}]   Y in [{Y.min():.3f}, {Y.max():.3f}]")

inputs: ['rho_c', 't_c', 'L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'p']
outputs: ['SF_c', 'SF_f', 'SF_int', 'SF_wr', 'C_delta', 't_sw', 'C_t', 'W_t']
train   7200 rows   X in [0.100, 0.900]   Y in [0.100, 0.900]
val      900 rows   X in [0.100, 0.900]   Y in [0.099, 0.894]
test     900 rows   X in [0.100, 0.900]   Y in [0.100, 1.082]


In [3]:
print("layers:          ", LAYERS)
print("parameter blocks:", SHAPES)
print("total parameters:", N_PARAMS)
assert N_PARAMS == 1287

layers:           [9, 28, 27, 8]
parameter blocks: [(28, 9), (28,), (27, 28), (27,), (8, 27), (8,)]
total parameters: 1287


In [4]:
w_random = np.random.default_rng(0).normal(size=N_PARAMS)
y, a1, a2 = forward(w_random, data["X_train"][:5])
print("input    ", data["X_train"][:5])
print("layer 1", a1.shape, " (values in [-1, 1]:", bool(np.all(np.abs(a1) <= 1)), ")")
print("layer 2", a2.shape)
print("output ", y.shape)



input     [[0.21851852 0.12223591 0.9        0.9        0.9        0.9
  0.1        0.9        0.74520458]
 [0.47037037 0.29910384 0.1        0.5        0.1        0.1
  0.5        0.5        0.380637  ]
 [0.54444444 0.422899   0.9        0.9        0.1        0.5
  0.9        0.9        0.78904618]
 [0.29259259 0.68955465 0.5        0.5        0.9        0.5
  0.5        0.1        0.85434484]
 [0.21851852 0.55855244 0.9        0.1        0.1        0.5
  0.9        0.1        0.20018962]]
layer 1 (5, 28)  (values in [-1, 1]: True )
layer 2 (5, 27)
output  (5, 8)


In [5]:
w0 = init_weights(SEED, data["X_train"])
print(f"initial MSE train {mse(w0, data['X_train'], data['Y_train']):.3e}   "
      f"val {mse(w0, data['X_val'], data['Y_val']):.3e}")


initial MSE train 3.114e+00   val 3.146e+00


In [6]:
np.savez(MODEL_DIR / "w0.npz", w0=w0, seed=SEED, layers = np.array(LAYERS))
print("saved", MODEL_DIR / "w0.npz")

saved /Users/aayushveturi/aayush_honours_thesis_B/models/ann_baseline/w0.npz
